# Scan analysis with DuckDB

This notebook queries every scan the Pluto has logged in `../data/` and answers questions
**across** scans: how many there are, how stable the noise floor is, and how busy each
radio band is over time. It never connects to the Pluto.

**How to run:** pick the kernel **Python (pluto)** (Kernel > Change Kernel), then run cells
top to bottom with `Shift+Enter`, or use Run > Run All Cells. Each section depends on
the ones above it, so if something errors, rerun from the top.

## 1. Setup

`sys.executable` shows which Python this kernel is running. It should end in
`envs\pluto\python.exe`. A notebook's working folder is the folder it's saved in
(`notebooks/`), so the project folder is one level up. Adding it to `sys.path` lets
us `import` the project's own scripts later, like `addpath` in MATLAB.

In [ ]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(sys.executable)

# Works whether Jupyter was started in notebooks/ or in the project folder.
PROJECT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
DATA = PROJECT / "data"
SCANS = (DATA / "scan_*.parquet").as_posix()   # DuckDB takes a glob pattern, / separators
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))           # like MATLAB addpath(PROJECT)

print(f"{len(list(DATA.glob('scan_*.parquet')))} scan files in {DATA}")

## 2. DuckDB: SQL directly on the Parquet files

In MATLAB you might `parquetread` every file and stack them into one big table. That
works for a few files, but 50 full scans is 9 million rows, and a day of logging is
far more.

DuckDB works differently. It's a small database engine that runs inside Python (no server),
and it can treat a whole folder of Parquet files as one table. A query only reads the
columns and rows it needs, does the math in fast compiled code, and returns only the
(small) result as a pandas DataFrame.

- `duckdb.connect()` with no file name makes an **in-memory** database: nothing is
  written to disk.
- A **view** is a saved query, not a copy of the data. `bins` below means "all rows of
  all scan files", and `filename=true` adds a column saying which file each row came from.
- `SET TimeZone = 'UTC'` makes timestamps display in UTC, the same as the file names.

In [ ]:
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
con.execute(f"CREATE OR REPLACE VIEW bins AS "
            f"SELECT * FROM read_parquet('{SCANS}', filename=true)")

con.sql("DESCRIBE bins").df()     # column names and types, like MATLAB's summary(T)

In [ ]:
con.sql("SELECT count(*) AS total_rows FROM bins").df()

## 3. Inventory: one row per scan

`GROUP BY filename` collapses each file's ~178,000 rows into one summary row (like
MATLAB's `groupsummary`). Some details:

- `count(psd_dbfs_hz)` counts only non-NULL values, so `n_bins - n_valid` is the number
  of masked DC bins.
- The **median** PSD estimates the noise floor. Most bins are noise, and a few strong
  signals barely move a median (they would pull a mean up).
- Gain isn't a column: it's stored in each file's **metadata**. `parquet_kv_metadata()`
  reads those key/value pairs, and a `JOIN` attaches the gain to each scan's row.
- The `regexp_extract` pulls the band (e.g. `325to3800MHz`) out of the file name.

In [ ]:
inventory = con.sql(rf'''
    WITH gains AS (
        SELECT file_name, decode(value) AS gain_db
        FROM parquet_kv_metadata('{SCANS}')
        WHERE decode(key) = 'gain_db'
    )
    SELECT min(b.timestamp)                                    AS scan_time,
           regexp_extract(b.filename, '_(\d+to\d+MHz)', 1)     AS band,
           any_value(g.gain_db)                                AS gain_db,
           count(*)                                            AS n_bins,
           count(b.psd_dbfs_hz)                                AS n_valid,
           round(median(b.psd_dbfs_hz), 2)                     AS noise_floor_db,
           round(max(b.psd_dbfs_hz), 1)                        AS peak_db,
           b.filename
    FROM bins b JOIN gains g ON b.filename = g.file_name
    GROUP BY b.filename
    ORDER BY scan_time
''').df()

inventory

In [ ]:
# pandas follow-up on the small result: how many scans per band and gain?
inventory.groupby(["band", "gain_db"]).size()

## 4. Noise floor over time

If the noise floor drifts between scans, comparing signal levels across scans is
misleading. Things that can make it move:
- **Temperature:** the AD9363's gain and noise figure change slightly as it warms up.
  The first scans after plugging in may differ from later ones.
- **Gain setting:** each dB of RX gain moves the noise floor in dBFS. All scans here
  used 40 dB, so that's not a factor (yet).
- **Real-world changes:** a strong nearby signal can raise the floor (by overloading
  the receiver or creating intermod products).

Only the full-range scans are compared, because a 2.4 GHz scan's "noise floor" is mostly
Wi-Fi. The x axis is scan number, not time, because the scans are weeks apart in
bursts, so a time axis would be mostly empty.

In [ ]:
full = inventory[inventory.band == "325to3800MHz"].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(full.index, full.noise_floor_db, "o-", ms=4)
ax.set_xlabel("full-range scan number (oldest = 0)")
ax.set_ylabel("median PSD (dBFS/Hz)")
ax.set_title("Noise floor per scan (median of all bins)")
ax.grid(alpha=0.3)

# Label the first scan of each day so you can see the logging sessions.
first_of_day = full.groupby(full.scan_time.dt.date).head(1)
for n, (i, t) in enumerate(zip(first_of_day.index, first_of_day.scan_time)):
    ax.axvline(i, color="gray", lw=0.5)
    ax.annotate(f"{t:%m-%d}", (i, 1), xycoords=("data", "axes fraction"),
                xytext=(2, -12 - 11 * (n % 3)),     # stagger so close labels don't overlap
                textcoords="offset points", fontsize=8)
plt.show()

print(f"spread: {full.noise_floor_db.min():.2f} to {full.noise_floor_db.max():.2f} dBFS/Hz "
      f"(std {full.noise_floor_db.std():.2f} dB)")

## 5. Band occupancy

**Occupancy** is the standard way spectrum monitors summarize "how busy is this band":
the fraction of frequency bins (in a band, in one scan) above a threshold. Here the
threshold is **noise floor + 10 dB**, per scan.

Why 10 dB? Each bin is a Welch average of ~127 segments, which makes the noise very
steady from bin to bin. Its standard deviation is about 4.34 dB / sqrt(127), roughly
0.4 dB. So noise alone almost never reaches even +3 dB. But the Pluto's noise floor isn't
perfectly flat across 325 MHz to 3.8 GHz (it varies by a couple of dB with frequency),
and +10 dB leaves room for that.

The bands are approximate US allocations, as a pandas DataFrame. `con.register` lets
SQL use a DataFrame as if it were a table. The `JOIN ... ON freq_hz BETWEEN lo AND hi`
assigns each bin to its band. That's a *range join*, which DuckDB is fast at.

In [ ]:
THRESHOLD_DB = 10    # occupied = more than this many dB above the scan's noise floor

bands = pd.DataFrame([
    # name,                low MHz, high MHz
    ("LTE 600 downlink",      617,  652),
    ("700 MHz downlink",      728,  768),
    ("850 cellular downlink", 869,  894),
    ("ISM 915",               902,  928),
    ("ADS-B 1090 (aircraft)", 1085, 1095),
    ("PCS downlink",         1930, 1995),
    ("AWS downlink",         2110, 2200),
    ("Wi-Fi 2.4 GHz",        2400, 2483.5),
    ("2.5 GHz (BRS/EBS)",    2496, 2690),
    ("C-band 5G (lower)",    3700, 3800),
], columns=["band", "lo_mhz", "hi_mhz"])
con.register("bands", bands)
bands

In [ ]:
occ = con.sql(f'''
    SELECT i.scan_time, k.band,
           100 * avg(CASE WHEN b.psd_dbfs_hz > i.noise_floor_db + {THRESHOLD_DB}
                          THEN 1.0 ELSE 0.0 END)   AS occupancy_pct
    FROM bins b
    JOIN inventory i ON b.filename = i.filename          -- per-scan noise floor
    JOIN bands k ON b.freq_hz BETWEEN k.lo_mhz * 1e6 AND k.hi_mhz * 1e6
    WHERE i.band = '325to3800MHz' AND b.psd_dbfs_hz IS NOT NULL
    GROUP BY ALL
    ORDER BY i.scan_time
''').df()

# Long table -> wide table: one row per band, one column per scan (like MATLAB unstack)
occ_wide = occ.pivot(index="band", columns="scan_time", values="occupancy_pct")
occ_wide = occ_wide.loc[bands.band]           # keep the bands in frequency order
occ_wide.mean(axis=1).round(1).rename("mean occupancy %").to_frame()

The same numbers as a heatmap, one column per scan. Steady bands (cellular base
stations transmit all the time) show as even stripes. Bursty ones like Wi-Fi and
ADS-B change from scan to scan, because each 15 MHz step only listens for ~3 ms.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(occ_wide.to_numpy(), aspect="auto", cmap="magma", vmin=0, vmax=100,
               interpolation="nearest")
ax.set_yticks(range(len(occ_wide)), occ_wide.index)
# One tick (and a white line) at the first scan of each logging day.
days = pd.Series(occ_wide.columns).dt.date
starts = np.flatnonzero(days != days.shift())
ax.set_xticks(starts, [f"{occ_wide.columns[c]:%m-%d}" for c in starts],
              rotation=45, ha="right")
for c in starts[1:]:
    ax.axvline(c - 0.5, color="white", lw=1)
ax.set_xlabel("first scan of each day (UTC); one column per scan")
fig.colorbar(im, ax=ax, label=f"% of bins > floor + {THRESHOLD_DB} dB")
ax.set_title("Band occupancy per full-range scan")
fig.tight_layout()
plt.show()

## 6. Reusing the project's scripts

`plot_scan.py` keeps its command-line code inside `main()`, behind
`if __name__ == "__main__":`. So importing it here runs nothing. It only makes its
functions available, much like calling a MATLAB function file from a script.

In [ ]:
from plot_scan import newest_scan, load_scan, plot_spectrum, gain_label

path = newest_scan(DATA)
df, meta = load_scan(path)
print(path.name, "| gain", gain_label(meta), "dB |", f"{len(df):,} bins")
fig = plot_spectrum(df, title=path.name)
plt.show()

In [ ]:
# A pandas filter on one scan (like logical indexing in MATLAB): the 10 strongest
# bins in the 2.4 GHz Wi-Fi band.
wifi = df[(df.freq_hz >= 2400e6) & (df.freq_hz <= 2483.5e6)]
wifi.nlargest(10, "psd_dbfs_hz")[["freq_hz", "psd_dbfs_hz", "center_hz"]]

## 7. Try it yourself

1. Change `THRESHOLD_DB` to 3 and to 20 and rerun section 5. Which bands change a lot,
   and which barely move? What does that say about how strong their signals are?
2. Add a band you expect to be empty (look at the max-hold plot from
   `waterfall_plot_scans.py` for a quiet stretch). Its occupancy should be close to 0%.
   If it isn't, why not?
3. Write a query that finds the scan where 2.4 GHz Wi-Fi was busiest.
   (Hint: `ORDER BY occupancy_pct DESC LIMIT 1` on `occ`.
   DuckDB can query the `occ` DataFrame directly.)
4. Log a longer run, e.g. `python pluto_rx_freq_scan.py --interval 60 --count 60` for an
   hour, then rerun this notebook. Does the noise floor drift while the Pluto warms up?
5. `waterfall_plot_scans.py` does all its work inside `main()`. Restructure it the way
   `plot_scan.py` was, so this notebook can call its waterfall as a function.